<a id="top">

# Table of Contents

- [Read csv file](#Read-csv-file)
- [Sorting on single column](#Sorting-on-single-column)
- [Sorting on multiple columns](#Sorting-on-multiple-columns)
- [Removing duplicates](#Removing-duplicates)
- [Creating new column](#Creating-new-column)
- [Renaming column(s)](#Renaming-column(s))
- [Data summarizations](#Data-summarizations)
- [Filtering](#Filtering)
- [Running total](#Running-total)
- [Ranking](#Ranking)
- [Joins](#Joins)
- [Exploding arrays](#Exploding-arrays)
- [RegEx](#RegEx)
- [Partitioning](#Partitioning)
- [Connecting to databases](#Connecting-to-databases)

#### PySpark official documentation navigation tip: in the left navigation panel, most things you'll want to look up are in the "[Spark SQL](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html)" top level menu.

## First some imports that the examples will use

[[Back to Top](#top)]

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import urllib.request

spark = (
    SparkSession.builder
    # Enable eager, interactive mode - typically do not do this with production code
    .config("spark.sql.repl.eagerEval.enabled", "True")
    .appName("PySpark Cheat Sheet")
    .getOrCreate()
)

## Read csv file

[[Back to Top](#top)]

Sample data can be obtained from [here](https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv).

There is a sample data set located at `data/cars.csv`.  Let's first inspect it and see if it has header row and how it is delimited using plain old terminal command `head`:

In [2]:
# Terminal commands can be executed within a code cell by preceding it with an exclamation symbol
!head data/cars.csv

Car;MPG;Cylinders;Displacement;Horsepower;Weight;Acceleration;Model;Origin
Chevrolet Chevelle Malibu;18.0;8;307.0;130.0;3504.;12.0;70;US
Buick Skylark 320;15.0;8;350.0;165.0;3693.;11.5;70;US
Plymouth Satellite;18.0;8;318.0;150.0;3436.;11.0;70;US
AMC Rebel SST;16.0;8;304.0;150.0;3433.;12.0;70;US
Ford Torino;17.0;8;302.0;140.0;3449.;10.5;70;US
Ford Galaxie 500;15.0;8;429.0;198.0;4341.;10.0;70;US
Chevrolet Impala;14.0;8;454.0;220.0;4354.;9.0;70;US
Plymouth Fury iii;14.0;8;440.0;215.0;4312.;8.5;70;US
Pontiac Catalina;14.0;8;455.0;225.0;4425.;10.0;70;US


We see that it does indeed have a header row and the columns are delimited by semi-colon.

In [3]:
cars = spark.read.csv('data/cars.csv', header=True, sep=";")

In [4]:
cars.show(5, truncate=False)

+-------------------------+----+---------+------------+----------+------+------------+-----+------+
|Car                      |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model|Origin|
+-------------------------+----+---------+------------+----------+------+------------+-----+------+
|Chevrolet Chevelle Malibu|18.0|8        |307.0       |130.0     |3504. |12.0        |70   |US    |
|Buick Skylark 320        |15.0|8        |350.0       |165.0     |3693. |11.5        |70   |US    |
|Plymouth Satellite       |18.0|8        |318.0       |150.0     |3436. |11.0        |70   |US    |
|AMC Rebel SST            |16.0|8        |304.0       |150.0     |3433. |12.0        |70   |US    |
|Ford Torino              |17.0|8        |302.0       |140.0     |3449. |10.5        |70   |US    |
+-------------------------+----+---------+------------+----------+------+------------+-----+------+
only showing top 5 rows


In [5]:
cars.printSchema()

root
 |-- Car: string (nullable = true)
 |-- MPG: string (nullable = true)
 |-- Cylinders: string (nullable = true)
 |-- Displacement: string (nullable = true)
 |-- Horsepower: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- Acceleration: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Origin: string (nullable = true)



#### Explicitly defining schema

In [6]:
schema = StructType([
    StructField("Car", StringType(),True),
    StructField("MPG", DoubleType(),True),
    StructField("Cylinders", IntegerType(),True),
    StructField("Displacement", DoubleType(), True),
    StructField("Horsepower", DoubleType(), True),
    StructField("Weight", DoubleType(), True),
    StructField("Acceleration", IntegerType(), True),
    StructField("Model", IntegerType(), True),
    StructField("Origin", StringType(), True)
])

In [7]:
cars = spark.read.csv('data/cars.csv', header=True, sep=";", schema=schema)

In [8]:
cars.printSchema()

root
 |-- Car: string (nullable = true)
 |-- MPG: double (nullable = true)
 |-- Cylinders: integer (nullable = true)
 |-- Displacement: double (nullable = true)
 |-- Horsepower: double (nullable = true)
 |-- Weight: double (nullable = true)
 |-- Acceleration: integer (nullable = true)
 |-- Model: integer (nullable = true)
 |-- Origin: string (nullable = true)



In [9]:
cars.show(5, truncate=False)

+-------------------------+----+---------+------------+----------+------+------------+-----+------+
|Car                      |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model|Origin|
+-------------------------+----+---------+------------+----------+------+------------+-----+------+
|Chevrolet Chevelle Malibu|18.0|8        |307.0       |130.0     |3504.0|NULL        |70   |US    |
|Buick Skylark 320        |15.0|8        |350.0       |165.0     |3693.0|NULL        |70   |US    |
|Plymouth Satellite       |18.0|8        |318.0       |150.0     |3436.0|NULL        |70   |US    |
|AMC Rebel SST            |16.0|8        |304.0       |150.0     |3433.0|NULL        |70   |US    |
|Ford Torino              |17.0|8        |302.0       |140.0     |3449.0|NULL        |70   |US    |
+-------------------------+----+---------+------------+----------+------+------------+-----+------+
only showing top 5 rows


## Sorting on single column

[[Back to Top](#top)]

In [10]:
(
    cars.sort(F.col("MPG").asc())
).show(10, truncate=False)

# Alternate method to refer to a dataframe column
# cars.sort(df.MPG.asc())

# Alternative methods if your column names has a space in it
# cars.sort(col("MPG").asc())

+--------------------------------+----+---------+------------+----------+------+------------+-----+------+
|Car                             |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model|Origin|
+--------------------------------+----+---------+------------+----------+------+------------+-----+------+
|Chevrolet Chevelle Concours (sw)|0.0 |8        |350.0       |165.0     |4142.0|NULL        |70   |US    |
|Citroen DS-21 Pallas            |0.0 |4        |133.0       |115.0     |3090.0|NULL        |70   |Europe|
|Ford Torino (sw)                |0.0 |8        |351.0       |153.0     |4034.0|NULL        |70   |US    |
|Plymouth Satellite (sw)         |0.0 |8        |383.0       |175.0     |4166.0|NULL        |70   |US    |
|AMC Rebel SST (sw)              |0.0 |8        |360.0       |175.0     |3850.0|NULL        |70   |US    |
|Ford Mustang Boss 302           |0.0 |8        |302.0       |140.0     |3353.0|NULL        |70   |US    |
|Volkswagen Super Beetle 117     |0.0

## Sorting on multiple columns

[[Back to Top](#top)]

In [11]:
(
    cars.sort(F.col("MPG").asc(), F.col("Displacement").desc())
).show(10, truncate=False)

# Alternative methods
# cars.sort(df.MPG.asc(), df.Displacement.desc())
# cars.sort(['MPG','Displacement'], ascending=[True, False]).show(10, truncate=False)

+--------------------------------+----+---------+------------+----------+------+------------+-----+------+
|Car                             |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model|Origin|
+--------------------------------+----+---------+------------+----------+------+------------+-----+------+
|Plymouth Satellite (sw)         |0.0 |8        |383.0       |175.0     |4166.0|NULL        |70   |US    |
|AMC Rebel SST (sw)              |0.0 |8        |360.0       |175.0     |3850.0|NULL        |70   |US    |
|Ford Torino (sw)                |0.0 |8        |351.0       |153.0     |4034.0|NULL        |70   |US    |
|Chevrolet Chevelle Concours (sw)|0.0 |8        |350.0       |165.0     |4142.0|NULL        |70   |US    |
|Ford Mustang Boss 302           |0.0 |8        |302.0       |140.0     |3353.0|NULL        |70   |US    |
|Citroen DS-21 Pallas            |0.0 |4        |133.0       |115.0     |3090.0|NULL        |70   |Europe|
|Saab 900s                       |0.0

## Removing duplicates

[[Back to Top](#top)]

#### Removing duplicates across _all_ [columns](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.distinct.html)

In [14]:
data = [
    ("James", "Sales", 3000),
    ("Michael", "Sales", 4600),
    ("Robert", "Sales", 4600),
    ("James", "Sales", 3000)
]

columns= ["employee_name", "department", "salary"]

# Create DataFrame
df = spark.createDataFrame(data = data, schema = columns)
df.printSchema()
df.show(truncate=False)

root
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|James        |Sales     |3000  |
|Michael      |Sales     |4600  |
|Robert       |Sales     |4600  |
|James        |Sales     |3000  |
+-------------+----------+------+



We see that James show up twice.  So let's delete the extra row or record.

In [13]:
distinctDF = df.distinct()
distinctDF.show(truncate=False)

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|James        |Sales     |3000  |
|Michael      |Sales     |4600  |
|Robert       |Sales     |4600  |
+-------------+----------+------+



#### Removing duplicates across _specific_ columns

[[Back to Top](#top)]

Let's say the goal is to remove duplicates based on multiple columns.  So in our original data:

In [15]:
df.show(truncate=False)

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|James        |Sales     |3000  |
|Michael      |Sales     |4600  |
|Robert       |Sales     |4600  |
|James        |Sales     |3000  |
+-------------+----------+------+



Let's say we want rows where department and salary are distinct.

In [16]:
dropDupesDF = df.drop_duplicates(["department","salary"])
dropDupesDF.show(truncate=False)

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|James        |Sales     |3000  |
|Michael      |Sales     |4600  |
+-------------+----------+------+



## Creating new column

[[Back to Top](#top)]

#### Creating new column based on partial text/string from another column

In [2]:
text = urllib.request.urlopen("https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv").read().decode("utf-8")
lines = text.splitlines()
cars = spark.read.csv(spark.sparkContext.parallelize(lines), header=True, inferSchema=True, sep=";")

In [3]:
cars = (
    cars
    .withColumn(
        "Make", F.get(F.split(F.col("Car"), " "), 0)
    )
    .withColumn(
        "Model", F.get(F.split(F.col("Car"), " "), 1)
    )
)

In [4]:
cars.show(5, truncate=False)

+-------------------------+----+---------+------------+----------+------+------------+---------+------+---------+
|Car                      |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model    |Origin|Make     |
+-------------------------+----+---------+------------+----------+------+------------+---------+------+---------+
|Chevrolet Chevelle Malibu|18.0|8        |307.0       |130.0     |3504  |12.0        |Chevelle |US    |Chevrolet|
|Buick Skylark 320        |15.0|8        |350.0       |165.0     |3693  |11.5        |Skylark  |US    |Buick    |
|Plymouth Satellite       |18.0|8        |318.0       |150.0     |3436  |11.0        |Satellite|US    |Plymouth |
|AMC Rebel SST            |16.0|8        |304.0       |150.0     |3433  |12.0        |Rebel    |US    |AMC      |
|Ford Torino              |17.0|8        |302.0       |140.0     |3449  |10.5        |Torino   |US    |Ford     |
+-------------------------+----+---------+------------+----------+------+------------+--

#### Using UDF to Create New Column

[[Back to Top](#top)]

In [5]:
data = [
    ("bacon", 4.0),
    ("pulled pork", 3.0),
    ("bacon", 12.0),
    ("pastrami", 6.0),
    ("corned beef", 7.5),
    ("bacon", 8.0),
    ("pastrami", 3.0),
    ("honey ham", 5.0),
    ("nova lox", 6.0),
  ]

In [6]:
schema = StructType([
    StructField("Food", StringType(),True),
    StructField("Ounces", DoubleType(),True),
])

In [7]:
df_food = spark.createDataFrame(data=data,schema=schema)

In [8]:
df_food.show()

+-----------+------+
|       Food|Ounces|
+-----------+------+
|      bacon|   4.0|
|pulled pork|   3.0|
|      bacon|  12.0|
|   pastrami|   6.0|
|corned beef|   7.5|
|      bacon|   8.0|
|   pastrami|   3.0|
|  honey ham|   5.0|
|   nova lox|   6.0|
+-----------+------+



In [9]:
def food2animal(column):
    if column == 'bacon':
        return 'pig'
    elif column == 'pulled pork':
        return 'pig'
    elif column == 'pastrami':
        return 'cow'
    elif column == 'corned beef':
        return 'cow'
    elif column == 'honey ham':
        return 'pig'
    else:
        return 'salmon'

In [10]:
food2animal_udf = F.udf(food2animal, StringType())

In [11]:
df_food_with_animal = df_food.withColumn("animal", food2animal_udf("Food"))

In [12]:
df_food_with_animal.show()

+-----------+------+------+
|       Food|Ounces|animal|
+-----------+------+------+
|      bacon|   4.0|   pig|
|pulled pork|   3.0|   pig|
|      bacon|  12.0|   pig|
|   pastrami|   6.0|   cow|
|corned beef|   7.5|   cow|
|      bacon|   8.0|   pig|
|   pastrami|   3.0|   cow|
|  honey ham|   5.0|   pig|
|   nova lox|   6.0|salmon|
+-----------+------+------+



#### Using `when` to duplicate IF-ELSE Logic To Create New Column

[[Back to Top](#top)]

In [13]:
data = [
    ("bacon", 4.0),
    ("pulled pork", 3.0),
    ("bacon", 12.0),
    ("pastrami", 6.0),
    ("corned beef", 7.5),
    ("bacon", 8.0),
    ("pastrami", 3.0),
    ("honey ham", 5.0),
    ("nova lox", 6.0),
  ]

schema = StructType([
    StructField("Food", StringType(),True),
    StructField("Ounces", DoubleType(),True),
])

df_food = spark.createDataFrame(data=data,schema=schema)

In [14]:
df_food.show()

+-----------+------+
|       Food|Ounces|
+-----------+------+
|      bacon|   4.0|
|pulled pork|   3.0|
|      bacon|  12.0|
|   pastrami|   6.0|
|corned beef|   7.5|
|      bacon|   8.0|
|   pastrami|   3.0|
|  honey ham|   5.0|
|   nova lox|   6.0|
+-----------+------+



In [15]:
df_food.withColumn(
    'Animal',
    F.when(F.col("Food") == 'bacon', 'pork')
    .when(F.col("Food") == 'pulled pork', 'pork')
    .when(F.col("Food") == 'pastrami', 'cow')
    .when(F.col("Food") == 'corned beef', 'cow')
    .when(F.col("Food") == 'honey ham', 'pig')
    .otherwise('salmon')
).show()

+-----------+------+------+
|       Food|Ounces|Animal|
+-----------+------+------+
|      bacon|   4.0|  pork|
|pulled pork|   3.0|  pork|
|      bacon|  12.0|  pork|
|   pastrami|   6.0|   cow|
|corned beef|   7.5|   cow|
|      bacon|   8.0|  pork|
|   pastrami|   3.0|   cow|
|  honey ham|   5.0|   pig|
|   nova lox|   6.0|salmon|
+-----------+------+------+



#### Using sql to create new column

[[Back to Top](#top)]

In [18]:
data = [
    ("bacon", 4.0),
    ("pulled pork", 3.0),
    ("bacon", 12.0),
    ("pastrami", 6.0),
    ("corned beef", 7.5),
    ("bacon", 8.0),
    ("pastrami", 3.0),
    ("honey ham", 5.0),
    ("nova lox", 6.0),
  ]

schema = StructType([
    StructField("Food", StringType(),True),
    StructField("Ounces", DoubleType(),True),
  ])

df_food = spark.createDataFrame(data=data,schema=schema)

In [19]:
df_food.show()

+-----------+------+
|       Food|Ounces|
+-----------+------+
|      bacon|   4.0|
|pulled pork|   3.0|
|      bacon|  12.0|
|   pastrami|   6.0|
|corned beef|   7.5|
|      bacon|   8.0|
|   pastrami|   3.0|
|  honey ham|   5.0|
|   nova lox|   6.0|
+-----------+------+



In [20]:
df_food.createOrReplaceTempView('food_table')
newDF = spark.sql(
    '''
    select *,
    case
    when Food = 'bacon' then 'pig'
    when Food = 'pulled pork' then 'pig'
    when Food = 'pastrami' then 'cow'
    when Food = 'corned beef' then 'cow'
    when Food = 'honey ham' then 'pig'
    else 'salmon' end as Animal from food_table
    '''
)
newDF.show()

+-----------+------+------+
|       Food|Ounces|Animal|
+-----------+------+------+
|      bacon|   4.0|   pig|
|pulled pork|   3.0|   pig|
|      bacon|  12.0|   pig|
|   pastrami|   6.0|   cow|
|corned beef|   7.5|   cow|
|      bacon|   8.0|   pig|
|   pastrami|   3.0|   cow|
|  honey ham|   5.0|   pig|
|   nova lox|   6.0|salmon|
+-----------+------+------+



**New in PySpark 3.4** - We no longer need to create temp view to directly query a dataframe using SQL:

In [22]:
spark.sql(
    "select * from {df}", df=newDF
).show()

+-----------+------+------+
|       Food|Ounces|Animal|
+-----------+------+------+
|      bacon|   4.0|   pig|
|pulled pork|   3.0|   pig|
|      bacon|  12.0|   pig|
|   pastrami|   6.0|   cow|
|corned beef|   7.5|   cow|
|      bacon|   8.0|   pig|
|   pastrami|   3.0|   cow|
|  honey ham|   5.0|   pig|
|   nova lox|   6.0|salmon|
+-----------+------+------+



#### Re-Arrange Columns using `select()`

[[Back to Top](#top)]

In [27]:
(
    cars.select('Make', 'Model', 'MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Origin')
).show()

+---------+----------+----+---------+------------+----------+------+------------+------+
|     Make|     Model| MPG|Cylinders|Displacement|Horsepower|Weight|Acceleration|Origin|
+---------+----------+----+---------+------------+----------+------+------------+------+
|Chevrolet|  Chevelle|18.0|        8|       307.0|     130.0| 3504.|        12.0|    US|
|    Buick|   Skylark|15.0|        8|       350.0|     165.0| 3693.|        11.5|    US|
| Plymouth| Satellite|18.0|        8|       318.0|     150.0| 3436.|        11.0|    US|
|      AMC|     Rebel|16.0|        8|       304.0|     150.0| 3433.|        12.0|    US|
|     Ford|    Torino|17.0|        8|       302.0|     140.0| 3449.|        10.5|    US|
|     Ford|   Galaxie|15.0|        8|       429.0|     198.0| 4341.|        10.0|    US|
|Chevrolet|    Impala|14.0|        8|       454.0|     220.0| 4354.|         9.0|    US|
| Plymouth|      Fury|14.0|        8|       440.0|     215.0| 4312.|         8.5|    US|
|  Pontiac|  Catalina

## Renaming column(s)

[[Back to Top](#top)]

#### Renaming a single column

In [30]:
(
    cars
    .withColumnRenamed(
        "Horsepower", "HP"
    )
).show()

+--------------------+----+---------+------------+-----+------+------------+----------+------+---------+
|                 Car| MPG|Cylinders|Displacement|   HP|Weight|Acceleration|     Model|Origin|     Make|
+--------------------+----+---------+------------+-----+------+------------+----------+------+---------+
|Chevrolet Chevell...|18.0|        8|       307.0|130.0|  3504|        12.0|  Chevelle|    US|Chevrolet|
|   Buick Skylark 320|15.0|        8|       350.0|165.0|  3693|        11.5|   Skylark|    US|    Buick|
|  Plymouth Satellite|18.0|        8|       318.0|150.0|  3436|        11.0| Satellite|    US| Plymouth|
|       AMC Rebel SST|16.0|        8|       304.0|150.0|  3433|        12.0|     Rebel|    US|      AMC|
|         Ford Torino|17.0|        8|       302.0|140.0|  3449|        10.5|    Torino|    US|     Ford|
|    Ford Galaxie 500|15.0|        8|       429.0|198.0|  4341|        10.0|   Galaxie|    US|     Ford|
|    Chevrolet Impala|14.0|        8|       454.0|220.0

#### Renaming multiple columns

[[Back to Top](#top)]

In [32]:
(
    cars
    .withColumnsRenamed(
        {
            "Cylinders": "Cyl",
            "Horsepower": "HP",
            "Displacement": "Displ",
        }
    )
).show()

+--------------------+----+---+-----+-----+------+------------+----------+------+---------+
|                 Car| MPG|Cyl|Displ|   HP|Weight|Acceleration|     Model|Origin|     Make|
+--------------------+----+---+-----+-----+------+------------+----------+------+---------+
|Chevrolet Chevell...|18.0|  8|307.0|130.0|  3504|        12.0|  Chevelle|    US|Chevrolet|
|   Buick Skylark 320|15.0|  8|350.0|165.0|  3693|        11.5|   Skylark|    US|    Buick|
|  Plymouth Satellite|18.0|  8|318.0|150.0|  3436|        11.0| Satellite|    US| Plymouth|
|       AMC Rebel SST|16.0|  8|304.0|150.0|  3433|        12.0|     Rebel|    US|      AMC|
|         Ford Torino|17.0|  8|302.0|140.0|  3449|        10.5|    Torino|    US|     Ford|
|    Ford Galaxie 500|15.0|  8|429.0|198.0|  4341|        10.0|   Galaxie|    US|     Ford|
|    Chevrolet Impala|14.0|  8|454.0|220.0|  4354|         9.0|    Impala|    US|Chevrolet|
|   Plymouth Fury iii|14.0|  8|440.0|215.0|  4312|         8.5|      Fury|    US

## Data summarizations

[[Back to Top](#top)]

In [16]:
cars.show(truncate=False)

+--------------------------------+----+---------+------------+----------+------+------------+----------+------+---------+
|Car                             |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model     |Origin|Make     |
+--------------------------------+----+---------+------------+----------+------+------------+----------+------+---------+
|Chevrolet Chevelle Malibu       |18.0|8        |307.0       |130.0     |3504  |12.0        |Chevelle  |US    |Chevrolet|
|Buick Skylark 320               |15.0|8        |350.0       |165.0     |3693  |11.5        |Skylark   |US    |Buick    |
|Plymouth Satellite              |18.0|8        |318.0       |150.0     |3436  |11.0        |Satellite |US    |Plymouth |
|AMC Rebel SST                   |16.0|8        |304.0       |150.0     |3433  |12.0        |Rebel     |US    |AMC      |
|Ford Torino                     |17.0|8        |302.0       |140.0     |3449  |10.5        |Torino    |US    |Ford     |
|Ford Galaxie 500       

#### Count of rows

In [17]:
cars.count()

406

#### Number of columns

In [19]:
len(cars.columns)

10

#### Counts by Groups within a Single Column

In [31]:
(
    cars.groupBy('Origin').count().withColumnRenamed('count', 'Count')
).show()

+------+-----+
|Origin|Count|
+------+-----+
|Europe|   73|
|    US|  254|
| Japan|   79|
+------+-----+



#### Aggregations

In [33]:
cars.groupBy(
    "Origin"
).agg(
    F.mean('MPG')
).show()

+------+------------------+
|Origin|          avg(MPG)|
+------+------------------+
|Europe|26.745205479452057|
|    US|19.688188976377948|
| Japan|30.450632911392397|
+------+------------------+



#### Creating pivot table

In [20]:
data = [
    ("dotNET", 2012, 10000),
    ("Java", 2012, 20000),
    ("dotNET", 2012, 5000),
    ("dotNET", 2013, 48000),
    ("Java", 2013, 30000)
]

columns = ["course", "year", "earnings"]
df = spark.createDataFrame(data, columns)

In [21]:
df.show()

+------+----+--------+
|course|year|earnings|
+------+----+--------+
|dotNET|2012|   10000|
|  Java|2012|   20000|
|dotNET|2012|    5000|
|dotNET|2013|   48000|
|  Java|2013|   30000|
+------+----+--------+



In [25]:
pivot_table = df.groupBy("year").pivot("course").sum("earnings")

pivot_table.show()

+----+-----+------+
|year| Java|dotNET|
+----+-----+------+
|2012|20000| 15000|
|2013|30000| 48000|
+----+-----+------+



#### If wanting to control order of columns, pass in list of column names as 2nd argument of `.pivot()` function

In [24]:
# Creating a pivot table
pivot_table = df.groupBy("year").pivot("course", ["dotNET", "Java"]).sum("earnings")

# Show the result
pivot_table.show()

+----+------+-----+
|year|dotNET| Java|
+----+------+-----+
|2012| 15000|20000|
|2013| 48000|30000|
+----+------+-----+



## Filtering

[[Back to Top](#top)]

First, let's rebuild our cars data set

In [43]:
text = urllib.request.urlopen("https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv").read().decode("utf-8")
lines = text.splitlines()
cars = spark.read.csv(spark.sparkContext.parallelize(lines), header=True, inferSchema=True, sep=";")

In [44]:
cars = (
    cars
    .withColumn(
        "Make", F.get(F.split(F.col("Car"), " "), 0)
    )
    .withColumn(
        "Model", F.get(F.split(F.col("Car"), " "), 1)
    )
)

In [45]:
cars.show(truncate=False)

+--------------------------------+----+---------+------------+----------+------+------------+----------+------+---------+
|Car                             |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model     |Origin|Make     |
+--------------------------------+----+---------+------------+----------+------+------------+----------+------+---------+
|Chevrolet Chevelle Malibu       |18.0|8        |307.0       |130.0     |3504  |12.0        |Chevelle  |US    |Chevrolet|
|Buick Skylark 320               |15.0|8        |350.0       |165.0     |3693  |11.5        |Skylark   |US    |Buick    |
|Plymouth Satellite              |18.0|8        |318.0       |150.0     |3436  |11.0        |Satellite |US    |Plymouth |
|AMC Rebel SST                   |16.0|8        |304.0       |150.0     |3433  |12.0        |Rebel     |US    |AMC      |
|Ford Torino                     |17.0|8        |302.0       |140.0     |3449  |10.5        |Torino    |US    |Ford     |
|Ford Galaxie 500       

In [47]:
cars.filter(F.col('Make')=='Chevrolet').show(5)

+--------------------+----+---------+------------+----------+------+------------+--------+------+---------+
|                 Car| MPG|Cylinders|Displacement|Horsepower|Weight|Acceleration|   Model|Origin|     Make|
+--------------------+----+---------+------------+----------+------+------------+--------+------+---------+
|Chevrolet Chevell...|18.0|        8|       307.0|     130.0|  3504|        12.0|Chevelle|    US|Chevrolet|
|    Chevrolet Impala|14.0|        8|       454.0|     220.0|  4354|         9.0|  Impala|    US|Chevrolet|
|Chevrolet Chevell...| 0.0|        8|       350.0|     165.0|  4142|        11.5|Chevelle|    US|Chevrolet|
|Chevrolet Monte C...|15.0|        8|       400.0|     150.0|  3761|         9.5|   Monte|    US|Chevrolet|
| Chevrolet Vega 2300|28.0|        4|       140.0|      90.0|  2264|        15.5|    Vega|    US|Chevrolet|
+--------------------+----+---------+------------+----------+------+------------+--------+------+---------+
only showing top 5 rows


In [48]:
cars.filter(F.col('Make').contains('Chev')).show(5)

+--------------------+----+---------+------------+----------+------+------------+--------+------+---------+
|                 Car| MPG|Cylinders|Displacement|Horsepower|Weight|Acceleration|   Model|Origin|     Make|
+--------------------+----+---------+------------+----------+------+------------+--------+------+---------+
|Chevrolet Chevell...|18.0|        8|       307.0|     130.0|  3504|        12.0|Chevelle|    US|Chevrolet|
|    Chevrolet Impala|14.0|        8|       454.0|     220.0|  4354|         9.0|  Impala|    US|Chevrolet|
|Chevrolet Chevell...| 0.0|        8|       350.0|     165.0|  4142|        11.5|Chevelle|    US|Chevrolet|
|Chevrolet Monte C...|15.0|        8|       400.0|     150.0|  3761|         9.5|   Monte|    US|Chevrolet|
|           Chevy C20|10.0|        8|       307.0|     200.0|  4376|        15.0|     C20|    US|    Chevy|
+--------------------+----+---------+------------+----------+------+------------+--------+------+---------+
only showing top 5 rows


When applying more than one boolean filtering, enclose each of your criterion in parenthesis

In [49]:
cars.filter(
    (F.col('Make').contains('Chev')) &
    (F.col('Cylinders') < 8)
).show()

+--------------------+----+---------+------------+----------+------+------------+--------+------+----------+
|                 Car| MPG|Cylinders|Displacement|Horsepower|Weight|Acceleration|   Model|Origin|      Make|
+--------------------+----+---------+------------+----------+------+------------+--------+------+----------+
| Chevrolet Vega 2300|28.0|        4|       140.0|      90.0|  2264|        15.5|    Vega|    US| Chevrolet|
|Chevrolet Chevell...|17.0|        6|       250.0|     100.0|  3329|        15.5|Chevelle|    US| Chevrolet|
| Chevrolet Vega (sw)|22.0|        4|       140.0|      72.0|  2408|        19.0|    Vega|    US| Chevrolet|
|      Chevrolet Vega|20.0|        4|       140.0|      90.0|  2408|        19.5|    Vega|    US| Chevrolet|
|Chevrolet Nova Cu...|16.0|        6|       250.0|     100.0|  3278|        18.0|    Nova|    US| Chevrolet|
|      Chevrolet Vega|21.0|        4|       140.0|      72.0|  2401|        19.5|    Vega|    US| Chevrolet|
|      Chevrolet No

## Running total

[[Back to Top](#top)]

**GOAL:** Running total / cumulative sum on MPG

Need to first define a "window" or subset of rows impacted by whatever calculation that is being directed at it

In [38]:
windowRunningTotal = Window.partitionBy(F.col("Origin")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

The `.rowsBetween(Window.unboundedPreceding, Window.currentRow)` defines the window frame: which rows, relative to the current row, the sum is allowed to look at.  For a running total you want “everything from the start of the partition up to and including this row,” and that’s exactly what rowsBetween(Window.unboundedPreceding, Window.currentRow) says.

- `Window.unboundedPreceding`: start at the first row of the partition
- `Window.currentRow`: stop at the current row



When to use `.rowsBetween(Window.unboundedPreceding, Window.currentRow)`:

- running total
- moving average


When not to use `.rowsBetween(Window.unboundedPreceding, Window.currentRow)`:

- rank()
- dense_rank()
- row_number()

In [39]:
(
    cars
    .withColumn(
        "cumulative_sum",
        F.sum(F.col("MPG")).over(windowRunningTotal)
    )
).show(truncate=False)

+----------------------------+----+---------+------------+----------+------+------------+-----+------+----------+--------------+
|Car                         |MPG |Cylinders|Displacement|Horsepower|Weight|Acceleration|Model|Origin|Make      |cumulative_sum|
+----------------------------+----+---------+------------+----------+------+------------+-----+------+----------+--------------+
|Citroen DS-21 Pallas        |0.0 |4        |133.0       |115.0     |3090  |17.5        |DS-21|Europe|Citroen   |0.0           |
|Volkswagen 1131 Deluxe Sedan|26.0|4        |97.0        |46.0      |1835  |20.5        |1131 |Europe|Volkswagen|26.0          |
|Peugeot 504                 |25.0|4        |110.0       |87.0      |2672  |17.5        |504  |Europe|Peugeot   |51.0          |
|Audi 100 LS                 |24.0|4        |107.0       |90.0      |2430  |14.5        |100  |Europe|Audi      |75.0          |
|Saab 99e                    |25.0|4        |104.0       |95.0      |2375  |17.5        |99e  |Eu

## Ranking

[[Back to Top](#top)]

#### Ranking using `dense_rank()`

See also, row_number() and rank() on how ties are handled

**GOAL:** Obtain top 3 ranked vehicles in each Origin with highest MPG

In [41]:
windowDenseRank = Window.partitionBy(F.col("Origin")).orderBy(F.col("MPG").desc())

In [42]:
(
    cars
    .withColumn(
        "rank",
        F.dense_rank().over(windowDenseRank)
    )
    .filter(
        F.col("rank").between(1,3)
    )
).show()

+--------------------+----+---------+------------+----------+------+------------+-------+------+----------+----+
|                 Car| MPG|Cylinders|Displacement|Horsepower|Weight|Acceleration|  Model|Origin|      Make|rank|
+--------------------+----+---------+------------+----------+------+------------+-------+------+----------+----+
|Volkswagen Rabbit...|44.3|        4|        90.0|      48.0|  2085|        21.7| Rabbit|Europe|Volkswagen|   1|
|   Volkswagen Pickup|44.0|        4|        97.0|      52.0|  2130|        24.6| Pickup|Europe|Volkswagen|   2|
|Volkswagen Dasher...|43.4|        4|        90.0|      48.0|  2335|        23.7| Dasher|Europe|Volkswagen|   3|
|           Mazda GLC|46.6|        4|        86.0|      65.0|  2110|        17.9|    GLC| Japan|     Mazda|   1|
| Honda Civic 1500 gl|44.6|        4|        91.0|      67.0|  1850|        13.8|  Civic| Japan|     Honda|   2|
|          Datsun 210|40.8|        4|        85.0|      65.0|  2110|        19.2|    210| Japan|

## Joins

[[Back to Top](#top)]

In [53]:
df_counts = cars.groupBy('Origin').count().withColumnRenamed('count', 'Count')

In [54]:
df_counts.show()

+------+-----+
|Origin|Count|
+------+-----+
|Europe|   73|
|    US|  254|
| Japan|   79|
+------+-----+



In [55]:
df_avgs = cars.groupBy(
    "Origin"
).agg(
    F.mean('MPG')
)

In [56]:
df_avgs.show()

+------+------------------+
|Origin|          avg(MPG)|
+------+------------------+
|Europe|26.745205479452057|
|    US|19.688188976377948|
| Japan|30.450632911392397|
+------+------------------+



In [57]:
df_avgs = df_avgs.withColumnRenamed('avg(MPG)', 'Avg')

In [58]:
df_avgs.show()

+------+------------------+
|Origin|               Avg|
+------+------------------+
|Europe|26.745205479452057|
|    US|19.688188976377948|
| Japan|30.450632911392397|
+------+------------------+



In [60]:
df_counts.join(df_avgs, df_counts.Origin == df_avgs.Origin, 'inner').select(df_counts.Origin, df_counts.Count, df_avgs.Avg).show()

+------+-----+------------------+
|Origin|Count|               Avg|
+------+-----+------------------+
|Europe|   73|26.745205479452057|
|    US|  254|19.688188976377948|
| Japan|   79|30.450632911392397|
+------+-----+------------------+



As you can see, we have done an inner join between two dataframes. The following joins are supported by PySpark:

- inner (default)
- cross
- outer
- full
- full_outer
- left
- left_outer
- right
- right_outer
- left_semi
- left_anti

## Exploding arrays

[[Back to Top](#top)]

Data set: movies data set from movielens

Question: Which genre of movie is the most popular?

In [3]:
!head data/movies.csv

movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy
6,Heat (1995),Action|Crime|Thriller
7,Sabrina (1995),Comedy|Romance
8,Tom and Huck (1995),Adventure|Children
9,Sudden Death (1995),Action


As can be seen by our sample data above, the problem we are facing is that a movie can have more than one genre, delimited by the pipe/"|" symbol.  So with the data in its current form, it would be difficult to obtain the counts by genre.  We will need to transform the data such that the counts by genre can be made.

Transformation plan:
- We will convert the genres into an array
- Then "explode" the genres such that each genre will be in its own row.  Consequently, this means the other columns will be repeating.
- Then perform a count by "genre"

In [4]:
movies = spark.read.csv('data/movies.csv', header=True, sep=",", inferSchema=True)

In [5]:
movies.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

In [12]:
from pyspark.sql.functions import col, split, explode

#### Answer: Drama

In [22]:
(
    movies
    .withColumn("genres_array", split(col("genres"), "\|"))
    .drop(col("genres"))
    .withColumn("genre", explode(col("genres_array")))
    .groupby(col("genre"))
    .count()
    .withColumnRenamed("genre", "Genre")
    .withColumnRenamed('count', 'Count')
    .sort(col("Count"), ascending=False)
    .filter(col("Genre") != '(no genres listed)')
).show(1)

+-----+-----+
|Genre|Count|
+-----+-----+
|Drama|25606|
+-----+-----+
only showing top 1 row



## RegEx

[[Back to Top](#top)]

In [23]:
from pyspark.sql.functions import regexp_extract

In [29]:
(
    movies
    .withColumn("year", regexp_extract(col("title"), r"\((\d{4})\)", 1))
)

movieId,title,genres,year
1,Toy Story (1995),Adventure|Animati...,1995
2,Jumanji (1995),Adventure|Childre...,1995
3,Grumpier Old Men ...,Comedy|Romance,1995
4,Waiting to Exhale...,Comedy|Drama|Romance,1995
5,Father of the Bri...,Comedy,1995
6,Heat (1995),Action|Crime|Thri...,1995
7,Sabrina (1995),Comedy|Romance,1995
8,Tom and Huck (1995),Adventure|Children,1995
9,Sudden Death (1995),Action,1995
10,GoldenEye (1995),Action|Adventure|...,1995


## Partitioning

[[Back to Top](#top)]

Work in Progress

## Connecting to databases

[[Back to Top](#top)]

[Link](https://spark.apache.org/docs/latest/api/python/user_guide/pandas_on_spark/from_to_dbms.html) to their documentation

The key thing to connect to relational databases using JDBC is to add `"spark.jars"` configuration.  You just need to ensure you download the required database driver and/or connector .jar files and reference their location or path.  You can connect to databases other than using JDBC, but since Java is the underlying technology behind Spark/PySpark, it is recommended that you use JDBC drivers for best performance and stability.

**NOTE:** Using `python-dotenv` library to load credentials from `.env` file

#### PostgreSQL

In [ ]:
import os

import pyspark
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()  # laod credentials from a .env file

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("Postgres")\
    .config("spark.jars", "C:\\Users\\some_user\\drivers\\jdbc\\postgresql\\postgresql-42.2.23.jar")\
    .getOrCreate()

In [ ]:
# Read in the Postgresql database credentials for DSN-less connection
PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DB = os.getenv("PG_DB")
PG_USER = os.getenv("PG_USERNAME")
PG_PWD = os.getenv("PG_PASSWORD")

In [ ]:
url = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}'
driver = 'org.postgresql.Driver'

In [ ]:
query = "SELECT CURRENT_DATE"

In [ ]:
jdbcDF = spark.read \
    .format("jdbc") \
    .option("driver", driver) \
    .option("url", url) \
    .option("user", PG_USER) \
    .option("password", PG_PWD) \
    .option("query", query) \
    .load()

In [ ]:
jdbcDF.show()

In [ ]:
spark.stop()

#### IBM DB2 LUW

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("SCOODS")\
    .config("spark.jars", "C:\\Users\\some_user\\drivers\\jdbc\\mainframe\\db2jcc.jar")\
    .getOrCreate()

In [ ]:
# Read in the DB2 LUW credentials for DSN-less connection
LUW_HOST = os.getenv("LUW_HOST")
LUW_PORT = os.getenv("LUW_PORT")
LUW_DB = os.getenv("LUW_DB")
LUW_USER = os.getenv("LUW_USERNAME")
LUW_PWD = os.getenv("LUW_PASSWORD")

In [ ]:
url = f'jdbc:db2://{LUW_HOST}:{LUW_PORT}/{LUW_DB}:useJDBC4ColumnNameAndLabelSemantics=false;'
driver = 'com.ibm.db2.jcc.DB2Driver'

In [ ]:
query = "SELECT CURRENT TIMESTAMP as DATETIME_NOW FROM SYSIBM.SYSDUMMY1"

In [ ]:
jdbcDF = spark.read \
    .format("jdbc") \
    .option("driver", driver) \
    .option("url", url) \
    .option("user", SCOODS_USER) \
    .option("password", SCOODS_PWD) \
    .option("query", query) \
    .load()

In [ ]:
jdbcDF.show(truncate=False)

In [ ]:
spark.stop()

#### Microsoft SQL Server

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("NAPS")\
    .config("spark.jars", "C:\\Users\\some_user\\drivers\\mssql_jdbc\\mssql-jdbc-9.4.0.jre8.jar")\
    .getOrCreate()

In [ ]:
# Read in the SQL Server database credentials for DSN-less connection
SSQL_HOST = os.getenv("MSSQL_HOST")
SSQL_PORT = os.getenv("MSSQL_PORT")
SSQL_DB = os.getenv("MSSQL_DB")

In [ ]:
url = f'jdbc:sqlserver://{SSQL_HOST}:{SSQL_PORT};databaseName={SSQL_DB};integratedSecurity=true'
driver = 'com.microsoft.sqlserver.jdbc.SQLServerDriver'

In [ ]:
query = "SELECT * from DimCarrier"

In [ ]:
jdbcDF = spark.read \
    .format("jdbc") \
    .option("driver", driver) \
    .option("url", url) \
    .option("query", query) \
    .load()

In [ ]:
jdbcDF.show(truncate=False)

In [ ]:
spark.stop()

#### IBM DB2 z/OS

Unfortunately, the IBM Java JRE does not work with Spark 3.x and thus, we will not be able to connect to mainframe DB2 z/OS platform.  Mainframe is too OLD!  Let it rest in peace.

#### Connecting to Snowflake

[Link](https://docs.snowflake.com/en/user-guide/spark-connector-install.html) to Snowflake's documentation on working with the PySpark connector

JDBC driver and Snowflake Spark Connector can be downloaded [here](https://search.maven.org/search?q=g:net.snowflake)

In [ ]:
sf_jdbc_driver = os.getenv("sf_jdbc_driver_path")
sf_spark_driver = os.getenv("sf_spark_driver_path")

In [ ]:
sf_account = os.getenv("sf_account")
sf_user = os.getenv("sf_username")
sf_database = os.getenv("sf_database")
sf_schema = os.getenv("sf_schema")
sf_role = os.getenv("sf_role")
sf_warehouse = os.getenv("sf_warehouse")
sf_authenticator = os.getenv("sf_authenticator")

In [ ]:
spark = (
    SparkSession.builder.master("local[*]")
    .appName("Snowflake_JDBC")
    .config("spark.jars", f"{sf_jdbc_driver},{sf_spark_driver}")
    .getOrCreate()
)

In [ ]:
SNOWFLAKE_SOURCE_NAME = "net.snowflake.spark.snowflake"

In [ ]:
# Snowflake connection parameters
sfparams = {
    "sfURL" : f"{sf_account}.snowflakecomputing.com",
    "sfUser" : sf_user,
    "sfPassword" : "your_password",  # Not applicable when using externalbrowser authenticator
    "sfDatabase" : sf_database,
    "sfSchema" : sf_schema,
    "sfRole" : sf_role,
    "sfWarehouse" : sf_warehouse,
    "sfAuthenticator" : sf_authenticator
}

In [ ]:
query = "SELECT CURRENT_DATE as my_date"

Snowflake query as PySpark dataframe

In [ ]:
#run custom query
df = (
    spark.read.format(SNOWFLAKE_SOURCE_NAME)
    .options(**sfparams)
    .option("query", query)
    .load()
)

In [ ]:
df.show()

Dataframe as Snowflake table

In [ ]:
(df
 .select("my_date").write.format(SNOWFLAKE_SOURCE_NAME)
 .options(**sfparams)
 .option("dbtable", "my_table")
 .mode("overwrite")
 .save()
)

In [ ]:
spark.stop()

#### Practical Scenarios

Let's say I have 10 CSV files, in which each CSV file contains stock price of a particular stock symbol.  I want to make a PySpark dataframe for each csv file or stock symbol.  But, we also need to create a new column in each dataframe that is the name of the CSV file without the file extension (".csv") which happens to be the stock symbol name.  Then concatenate the dataframes into a single dataframe.  How do I do this with PySpark?

In [17]:
from pyspark.sql import SparkSession
from functools import reduce
from pathlib import Path
from pyspark.sql.functions import lit

# Initialize a SparkSession
spark = SparkSession.builder.appName("ConcatenateCSVFiles").getOrCreate()

# Directory containing the CSV files
csv_directory = Path("stocks/")

# Get a list of CSV file paths in the directory
csv_files = [file for file in csv_directory.iterdir() if file.suffix == ".csv"]

# Create an empty list to store DataFrames
dataframes = []

# Iterate through the CSV files
for csv_file in csv_files:
    # Load the CSV file into a DataFrame
    df = spark.read.csv(str(csv_file), header=True, inferSchema=True)
    
    # Extract the file name without the extension
    file_name = csv_file.stem
    
    # Add a new column with the file name
    df = df.withColumn("Stock Symbol", lit(file_name))
    
    # Append the DataFrame to the list
    dataframes.append(df)

# Concatenate the DataFrames into a single DataFrame
concatenated_df = reduce(lambda df1, df2: df1.unionAll(df2), dataframes)

# Show the result
concatenated_df.show()

# Save the concatenated DataFrame as a CSV file, not partition, but single csv file
concatenated_df.coalesce(1).write.mode('overwrite').option("header", "true").csv("stocks/concatenated_data.csv")

pandas_df = concatenated_df.toPandas()
pandas_df.to_csv('stocks/pandas_df.csv', index=False)

# Stop the SparkSession when done
spark.stop()

+----------+------------------+------------------+------------------+------------------+-------------------+---------+------------+
|      Date|              Open|              High|               Low|             Close|          Adj Close|   Volume|Stock Symbol|
+----------+------------------+------------------+------------------+------------------+-------------------+---------+------------+
|1980-12-12|0.5133928656578064|          0.515625|0.5133928656578064|0.5133928656578064|0.40678155422210693|117258400|        AAPL|
|1980-12-15|0.4888392984867096|0.4888392984867096|0.4866071343421936|0.4866071343421936|  0.385558158159256| 43971200|        AAPL|
|1980-12-16|          0.453125|          0.453125|0.4508928656578064|0.4508928656578064| 0.3572602868080139| 26432000|        AAPL|
|1980-12-17|0.4620535671710968|0.4642857015132904|0.4620535671710968|0.4620535671710968| 0.3661033511161804| 21610400|        AAPL|
|1980-12-18|0.4754464328289032|0.4776785671710968|0.4754464328289032|0.47544